# Matching Tune - experiments

Task 7, Week 3, Phase 2. Walking through the matching engine from the dumb baseline
up to the tuned model that actually ships. This is the exploration notebook -
the real code lives in `src/` and gets imported here, not rewritten.

Order of work:
1. Look at the data
2. Baseline (skill overlap)
3. TF-IDF + cosine similarity
4. Tune the blend on validation, check it holds on test
5. One full demo walkthrough (student -> ranked jobs -> why)


In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

from matching import TfidfMatcher, parse_skills, baseline_score, explain_match, hybrid_score, rank_jobs
from evaluate import precision_recall_fpr_at_k, tune_alpha, baseline_only_precision_recall_fpr, tfidf_only_precision_recall_fpr

pd.set_option('display.max_colwidth', 80)


## 1. The data

Synthetic, but real-shaped: 10 roles, 80 jobs, 300 students with noisy/partial skill lists (see `src/data_gen.py` for how it's built).

In [ ]:
jobs_df = pd.read_csv('../data/jobs.csv')
students_df = pd.read_csv('../data/students.csv')

print(jobs_df.shape, students_df.shape)
jobs_df.head()


In [ ]:
students_df.head()


In [ ]:
students_df['split'].value_counts()


## 2. Baseline: skill overlap

`matched_skills / required_skills`. Nothing clever, just intersection over the job's required set.
Every later number in this notebook is judged against this.


In [ ]:
sample_student = parse_skills('Python,SQL,Machine Learning')
sample_job = parse_skills('Python,SQL')

print(sample_student, '->', sample_job)
print('overlap score:', baseline_score(sample_student, sample_job))


Matches the worked example from the study guide (2/2 skills matched = 100%).

## 3. TF-IDF + cosine similarity

Skill lists -> short text docs -> TF-IDF vectors -> cosine similarity against every job.
`TfidfMatcher` in `matching.py` fits the vectorizer once on the job pool, then scores any student against all 80 jobs in one shot.


In [ ]:
matcher = TfidfMatcher(jobs_df)

demo_skills = parse_skills('Python,SQL,Machine Learning')
sims = matcher.score_all(demo_skills)

top5_idx = np.argsort(sims)[::-1][:5]
jobs_df.iloc[top5_idx][['job_id', 'title', 'company', 'required_skills']].assign(tfidf_score=sims[top5_idx].round(3))


## 4. Tuning the blend (and checking it actually generalises)

`hybrid_score = alpha * tfidf + (1 - alpha) * overlap`. We sweep alpha on the **validation**
split only and pick whatever gives the best precision@5, then check train/val/test all together
so we're not fooling ourselves with a number that only looks good on the set we tuned on.


In [ ]:
train_df = students_df[students_df['split'] == 'train']
val_df = students_df[students_df['split'] == 'val']
test_df = students_df[students_df['split'] == 'test']

best_alpha, val_precision = tune_alpha(matcher, val_df)
print(f'best alpha on val: {best_alpha} (precision@5 = {val_precision})')


In [ ]:
rows = []
for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    b = baseline_only_precision_recall_fpr(matcher, split_df)
    t = tfidf_only_precision_recall_fpr(matcher, split_df)
    h = precision_recall_fpr_at_k(matcher, split_df, alpha=best_alpha)
    for name, m in [('baseline', b), ('tfidf', t), ('tuned_hybrid', h)]:
        rows.append({'model': name, 'split': split_name, **m})

results_df = pd.DataFrame(rows)
results_df


Precision and recall both move up going baseline -> tfidf, and the gap holds up on the
test split (not just train/val) - so this isn't a case of tuning to the demo data and
watching it collapse later. FPR is low everywhere because there are only ~8 truly relevant
jobs out of 80 per student, so most of the pool is correctly excluded by both models.


## 5. Demo walkthrough - one student, end to end

This is the version to actually show live: pick one student, show their skills,
show the ranked jobs, and explain the top result in plain English.


In [ ]:
demo_student = students_df.sample(1, random_state=7).iloc[0]
print('Student:', demo_student['name'])
print('Skills:', demo_student['skills'])
print('Verified avg skill score:', demo_student['avg_skill_score'])


In [ ]:
ranked = rank_jobs(parse_skills(demo_student['skills']), matcher, alpha=best_alpha, top_n=5)

for rank, r in enumerate(ranked, start=1):
    print(f"{rank}. {r['title']} @ {r['company']} -> score {r['score']}")
    print(f"   matched: {r['explanation']['matched_skills']}")
    print(f"   missing: {r['explanation']['missing_skills']}")


That's the full pipeline: baseline -> tfidf -> tuned hybrid -> evaluated on a held-out
test split -> persisted as `models/matching_model.pkl` -> served from `api/app.py`.

Next: hook this into the live pay-per-apply flow and watch precision@5 as a guardrail
metric after the pricing change goes out (per the study guide's note on generalisation).
